# Implied Volatility Surface Imputation
**Competition:** Finance Club IIT Roorkee — Open Projects 2026  
**Task:** Predict 5460 missing implied volatility values across 28 NIFTY 50 option contracts  
**Metric:** Mean Squared Error (lower is better)  
**Best public score:** 0.0000388978

---

## Approach Summary
Six cross-sectional smile-fitting methods are blended with QP-optimised weights, using
**stratified cross-validation** that mirrors the actual missing distribution and
**3 TTE buckets** to handle the distinct expiry-day regime separately.

| Method | Description |
|--------|-------------|
| `poly2` | Quadratic polynomial: IV ~ a + bK + cK² |
| `poly2_var` | Quadratic in variance space: IV² ~ a + bK + cK² |
| `PCHIP` | Monotone cubic spline (PCHIP), extrapolated |
| `adaptive` | PCHIP for interior strikes; poly2 for wings |
| `logwing` | PCHIP interior; log-linear wing extrapolation |
| `totvar` | PCHIP on total variance w = IV² × TTE (bounded near expiry) |

Key insight: fitting in total-variance space w = IV²×TTE prevents the near-expiry IV spike
(IV → 5+ in the final trading hour) from blowing up cross-sectional fits.


## 0. Imports & Configuration

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
from scipy.optimize import minimize as _minimize
from sklearn.metrics import mean_squared_error

DATASET_PATH  = 'dataset.csv'
FILLED_PATH   = 'filled_dataset.csv'
SUBMISSION_PATH = 'submission.csv'
EXPIRY = pd.Timestamp('2026-01-27 15:30')
TRADING_MIN_PER_YEAR = 252 * 390   # 98,280 trading minutes per year
TTE_SPLIT  = 5 * 390               # 1950 min = 5 trading days
TTE_FINAL  = 60                    # 60 min  = last trading hour of expiry day
SEED = 42


## 1. Load Data & Parse Metadata

In [ ]:
df_raw = pd.read_csv(DATASET_PATH)
df_raw['datetime'] = pd.to_datetime(df_raw['datetime'], format='%d-%m-%Y %H:%M')

option_cols = [c for c in df_raw.columns if re.match(r'NIFTY', c)]
ce_cols  = sorted([c for c in option_cols if 'CE' in c])
pe_cols  = sorted([c for c in option_cols if 'PE' in c])
ce_strikes = [int(re.search(r'(\d{5})CE', c).group(1)) for c in ce_cols]
pe_strikes = [int(re.search(r'(\d{5})PE', c).group(1)) for c in pe_cols]

df_raw['tte_minutes'] = (EXPIRY - df_raw['datetime']).dt.total_seconds() / 60

print(f'Dataset shape : {df_raw.shape}')
print(f'Option columns: {len(option_cols)} ({len(ce_cols)} CE, {len(pe_cols)} PE)')
print(f'Missing values: {df_raw[option_cols].isna().sum().sum()} / {df_raw[option_cols].size}')
print(f'Date range    : {df_raw["datetime"].min().date()} to {df_raw["datetime"].max().date()}')
print(f'TTE range     : [{df_raw["tte_minutes"].min():.0f}, {df_raw["tte_minutes"].max():.0f}] min')


## 2. Exploratory Data Analysis

In [ ]:
# 2.1  Missingness heatmap — shows which contracts and timestamps have missing IVs
fig, ax = plt.subplots(figsize=(16, 4))
miss = df_raw[option_cols].isna().astype(int)
ax.imshow(miss.T, aspect='auto', cmap='Reds', interpolation='none')
ax.set_xlabel('Timestamp index'); ax.set_ylabel('Option contract')
ax.set_title('Missingness heatmap (red = missing)')
plt.tight_layout(); plt.show()

print('Missing rate per contract (top 5):')
per_col = df_raw[option_cols].isna().mean().sort_values(ascending=False)
print(per_col.head(5).to_string())


In [ ]:
# 2.2  IV smile shape: normal day vs expiry day
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, ts_label in zip(axes, ['2026-01-14', '2026-01-27']):
    sample = df_raw[df_raw['datetime'].dt.date == pd.Timestamp(ts_label).date()].iloc[20]
    for cols, strikes, label in [(ce_cols, ce_strikes, 'CE'), (pe_cols, pe_strikes, 'PE')]:
        ivs = [sample[c] for c in cols]
        obs_s = [s for s,v in zip(strikes,ivs) if not pd.isna(v)]
        obs_v = [v for v in ivs if not pd.isna(v)]
        ax.plot(obs_s, obs_v, 'o-', label=label)
    ax.set_title(f'IV smile — {ts_label}'); ax.set_xlabel('Strike'); ax.set_ylabel('IV')
    ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# 2.3  IV time-series for near-ATM CE — shows expiry spike
atm_col = ce_cols[len(ce_cols)//2]
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(df_raw['datetime'], df_raw[atm_col], lw=0.8)
ax.set_title(f'IV time-series: {atm_col}')
ax.set_xlabel('Datetime'); ax.set_ylabel('Implied Volatility')
plt.tight_layout(); plt.show()
print(f'Note: Jan 27 spike (expiry day) — IV rises from ~0.1 to 5+ as options approach expiration.')


## 3. Fill Methods

Six cross-sectional methods fit the IV smile at each timestamp independently using available observed strikes.

In [ ]:
# 4.1  Core fill functions

def fill_poly2(row_iv, strikes):
    """Quadratic polynomial IV ~ a + bK + cK^2."""
    obs = [(s,v) for s,v in zip(strikes,row_iv.values) if not np.isnan(v)]
    if len(obs) < 2: return row_iv
    ks,vs = zip(*obs); c = np.polyfit(ks, vs, min(2, len(obs)-1))
    if len(c) == 2: c = [0] + list(c)
    r = row_iv.copy()
    for s in strikes:
        if np.isnan(row_iv[s]): r[s] = max(np.polyval(c,s), 1e-4)
    return r

def fill_poly2_var(row_iv, strikes):
    """Quadratic in variance space: IV^2 ~ a + bK + cK^2."""
    obs = [(s,v) for s,v in zip(strikes,row_iv.values) if not np.isnan(v)]
    if len(obs) < 2: return fill_poly2(row_iv, strikes)
    ks,vs = zip(*obs); ws = [v**2 for v in vs]
    c = np.polyfit(ks, ws, min(2, len(obs)-1))
    if len(c) == 2: c = [0] + list(c)
    r = row_iv.copy()
    for s in strikes:
        if np.isnan(row_iv[s]): r[s] = max(np.sqrt(max(np.polyval(c,s), 1e-8)), 1e-4)
    return r

def fill_pchip(row_iv, strikes):
    """PCHIP monotone cubic spline with extrapolation."""
    obs = [(s,v) for s,v in zip(strikes,row_iv.values) if not np.isnan(v)]
    if len(obs) < 2: return fill_poly2(row_iv, strikes)
    ks,vs = zip(*obs)
    fn = PchipInterpolator(ks, vs, extrapolate=True) if len(obs)>=3 else lambda x: float(np.interp(x,ks,vs))
    r = row_iv.copy()
    for s in strikes:
        if np.isnan(row_iv[s]): r[s] = max(float(fn(s)), 1e-4)
    return r

def fill_adaptive(row_iv, strikes):
    """PCHIP for interior strikes; poly2 for wing extrapolation."""
    obs_s = [s for s,v in zip(strikes,row_iv.values) if not np.isnan(v)]
    if not obs_s: return row_iv
    lo,hi = min(obs_s), max(obs_s); r = row_iv.copy()
    interior = [s for s in strikes if np.isnan(row_iv[s]) and lo<=s<=hi]
    wings    = [s for s in strikes if np.isnan(row_iv[s]) and (s<lo or s>hi)]
    if interior:
        rp = fill_pchip(row_iv, strikes)
        for s in interior: r[s] = rp[s]
    if wings:
        rp = fill_poly2(row_iv, strikes)
        for s in wings: r[s] = rp[s]
    return r

def fill_logwing(row_iv, strikes):
    """PCHIP interior + log-linear wing extrapolation.
    Slope estimated from observed strikes within ±100 of the boundary; clipped to ±0.02/unit.
    Ensures geometric extrapolation that never goes negative."""
    obs = [(s,v) for s,v in zip(strikes,row_iv.values) if not np.isnan(v) and v>0]
    if len(obs) < 2: return fill_poly2(row_iv, strikes)
    obs_s = [s for s,v in obs]; lo,hi = min(obs_s), max(obs_s)
    r = fill_pchip(row_iv, strikes)
    lo_iv = row_iv[lo]; hi_iv = row_iv[hi]
    inner_lo = sorted([(s,v) for s,v in obs if lo<=s<=lo+100])
    inner_hi = sorted([(s,v) for s,v in obs if hi-100<=s<=hi])
    sl = np.clip((np.log(inner_lo[-1][1])-np.log(inner_lo[0][1]))/(inner_lo[-1][0]-inner_lo[0][0]+1e-9),-0.02,0.02) if len(inner_lo)>=2 else 0.0
    sh = np.clip((np.log(inner_hi[-1][1])-np.log(inner_hi[0][1]))/(inner_hi[-1][0]-inner_hi[0][0]+1e-9),-0.02,0.02) if len(inner_hi)>=2 else 0.0
    for s in strikes:
        if np.isnan(row_iv[s]):
            if s < lo:  r[s] = max(lo_iv * np.exp(sl*(s-lo)), 1e-4)
            elif s > hi: r[s] = max(hi_iv * np.exp(sh*(s-hi)), 1e-4)
    return r

def fill_totvar(row_iv, strikes, tte_minutes):
    """PCHIP in total-variance space: w = IV^2 * TTE.
    Near expiry, IV->inf but w stays bounded, making the fit stable
    in the final trading hour when IV can exceed 5."""
    obs = [(s,v) for s,v in zip(strikes,row_iv.values) if not np.isnan(v) and v>0]
    if len(obs) < 2: return fill_poly2(row_iv, strikes)
    T = max(tte_minutes / TRADING_MIN_PER_YEAR, 1e-7)
    ks = [s for s,v in obs]; ws = [v**2*T for s,v in obs]
    fn = PchipInterpolator(ks, ws, extrapolate=True) if len(obs)>=3 else lambda x: float(np.interp(x,ks,ws))
    r = row_iv.copy()
    for s in strikes:
        if np.isnan(row_iv[s]):
            r[s] = max(np.sqrt(max(float(fn(s)), 1e-10) / T), 1e-4)
    return r

def apply_fill(df, fn):
    out = df.copy()
    for cols,strikes in [(ce_cols,ce_strikes),(pe_cols,pe_strikes)]:
        piv = out[cols].copy(); piv.columns = strikes
        for idx in piv.index: piv.loc[idx] = fn(piv.loc[idx], strikes)
        out[cols] = piv.values
    return out

def apply_fill_tte(df, fn):
    """Like apply_fill but also passes tte_minutes to fn."""
    out = df.copy()
    for cols,strikes in [(ce_cols,ce_strikes),(pe_cols,pe_strikes)]:
        piv = out[cols].copy(); piv.columns = strikes
        for idx in piv.index:
            piv.loc[idx] = fn(piv.loc[idx], strikes, df.loc[idx,'tte_minutes'])
        out[cols] = piv.values
    return out

print('All 6 fill methods defined.')


## 4. Validation Strategy — Stratified CV

**Why stratified?** A flat 10% random mask underrepresents OTM wing contracts (24000PE, 26400CE)
that are ~22% missing in the actual test. Masking each contract at its *actual* missing rate gives
a CV distribution that closely matches the real test, so QP weights generalise better.


In [ ]:
# 4.1  Stratified CV mask — each contract masked at its actual missing rate (5%–50%)
np.random.seed(SEED)
actual_rate = {col: df_raw[col].isna().mean() for col in option_cols}

df_masked = df_raw.copy()
masked = []
for col in option_cols:
    obs_idx = df_raw.index[~df_raw[col].isna()].tolist()
    rate    = np.clip(actual_rate[col], 0.05, 0.50)
    n_mask  = max(1, int(len(obs_idx) * rate))
    chosen  = np.random.choice(obs_idx, size=n_mask, replace=False)
    for idx in chosen:
        df_masked.loc[idx, col] = np.nan
        masked.append((idx, col))

true_ivs = np.array([df_raw.loc[i,c] for i,c in masked])
tte_vals  = np.array([df_raw.loc[i,'tte_minutes'] for i,c in masked])

final_idx  = np.where(tte_vals <= TTE_FINAL)[0]
mid_idx    = np.where((tte_vals > TTE_FINAL) & (tte_vals <= TTE_SPLIT))[0]
normal_idx = np.where(tte_vals > TTE_SPLIT)[0]

print(f'Stratified CV: {len(masked):,} positions')
print(f'  normal        (TTE > 1950) : {len(normal_idx)}')
print(f'  mid-expiry    (60<TTE<=1950): {len(mid_idx)}')
print(f'  final-stretch (TTE <= 60)  : {len(final_idx)}')
print(f'Missing-rate range: [{min(actual_rate.values()):.1%}, {max(actual_rate.values()):.1%}]')


In [ ]:
# 4.2  Run all 6 fill methods on the masked dataset
def get_preds(df_f): return np.array([df_f.loc[i,c] for i,c in masked])

print('Running CV fills...')
p_p2  = get_preds(apply_fill(df_masked.copy(), fill_poly2))
p_var = get_preds(apply_fill(df_masked.copy(), fill_poly2_var))
p_ph  = get_preds(apply_fill(df_masked.copy(), fill_pchip))
p_adp = get_preds(apply_fill(df_masked.copy(), fill_adaptive))
p_lgw = get_preds(apply_fill(df_masked.copy(), fill_logwing))
p_tv  = get_preds(apply_fill_tte(df_masked.copy(), fill_totvar))
print('Done.')

KEYS  = ['p2','var','ph','adp','lgw','tv']
preds = {'p2':p_p2,'var':p_var,'ph':p_ph,'adp':p_adp,'lgw':p_lgw,'tv':p_tv}

print(f'{"Method":<10} {"Normal":>12} {"Mid-exp":>12} {"Final":>12}')
print('-' * 48)
for k,p in preds.items():
    mn = mean_squared_error(true_ivs[normal_idx], p[normal_idx])
    mm = mean_squared_error(true_ivs[mid_idx],    p[mid_idx])
    mf = mean_squared_error(true_ivs[final_idx],  p[final_idx])
    print(f'{k:<10} {mn:>12.6f} {mm:>12.6f} {mf:>12.6f}')


## 5. QP Weight Optimisation

Non-negative sum-to-1 weights found by SLSQP (25 Dirichlet restarts) separately for each TTE bucket.

In [ ]:
def qp(preds_dict, true_vals, n_restarts=25):
    ks = list(preds_dict.keys()); X = np.stack([preds_dict[k] for k in ks], axis=1)
    def obj(w): return mean_squared_error(true_vals, X @ w)
    cons   = [{'type':'eq','fun':lambda w:w.sum()-1}]
    bounds = [(0,1)] * len(ks)
    best   = None
    np.random.seed(SEED)
    for _ in range(n_restarts):
        w0  = np.random.dirichlet(np.ones(len(ks)))
        res = _minimize(obj, w0, method='SLSQP', bounds=bounds, constraints=cons,
                        options={'ftol':1e-12,'maxiter':2000})
        if best is None or res.fun < best.fun: best = res
    return dict(zip(ks, best.x)), best.fun

print('Optimising QP weights per bucket...')
W_N, _ = qp({k: preds[k][normal_idx] for k in KEYS}, true_ivs[normal_idx])
W_M, _ = qp({k: preds[k][mid_idx]    for k in KEYS}, true_ivs[mid_idx])
W_F, _ = qp({k: preds[k][final_idx]  for k in KEYS}, true_ivs[final_idx]) \
          if len(final_idx) >= 20 else (W_M.copy(), None)

for lbl, W in [('Normal       ', W_N), ('Mid-expiry   ', W_M), ('Final-stretch', W_F)]:
    print(f'{lbl}: {" ".join(f"{k}={v:.3f}" for k,v in W.items())}')

p_blend = np.empty_like(true_ivs)
p_blend[normal_idx] = np.stack([preds[k][normal_idx] for k in KEYS],axis=1) @ np.array([W_N[k] for k in KEYS])
p_blend[mid_idx]    = np.stack([preds[k][mid_idx]    for k in KEYS],axis=1) @ np.array([W_M[k] for k in KEYS])
p_blend[final_idx]  = np.stack([preds[k][final_idx]  for k in KEYS],axis=1) @ np.array([W_F[k] for k in KEYS])
print(f'\nBlend CV MSE: {mean_squared_error(true_ivs, p_blend):.8f}')


## 6. Build Full Surface & Generate Submission

In [ ]:
# Build all 6 fills on the complete dataset (no masking)
print('Building 6 fills on full dataset...')
fills = {
    'p2' : apply_fill(df_raw.copy(), fill_poly2),
    'var' : apply_fill(df_raw.copy(), fill_poly2_var),
    'ph'  : apply_fill(df_raw.copy(), fill_pchip),
    'adp' : apply_fill(df_raw.copy(), fill_adaptive),
    'lgw' : apply_fill(df_raw.copy(), fill_logwing),
    'tv'  : apply_fill_tte(df_raw.copy(), fill_totvar),
}
print('Done.')


In [ ]:
# Apply 3-bucket QP blend to fill all missing positions -> filled_dataset.csv
df_filled = df_raw.copy()
for col in option_cols:
    mm = df_raw[col].isna()
    if not mm.any(): continue
    for idx in df_raw.index[mm]:
        tte  = df_raw.loc[idx, 'tte_minutes']
        W    = W_F if tte <= TTE_FINAL else (W_M if tte <= TTE_SPLIT else W_N)
        vals = np.array([fills[k].loc[idx, col] for k in KEYS])
        df_filled.loc[idx, col] = max(float(np.dot(vals, [W[k] for k in KEYS])), 1e-4)

assert df_filled[option_cols].isna().sum().sum() == 0, 'Still missing values!'
df_filled['datetime'] = df_filled['datetime'].dt.strftime('%d-%m-%Y %H:%M')
df_filled.to_csv(FILLED_PATH, index=False)
print(f'Saved {FILLED_PATH}')


In [ ]:
# Generate submission.csv: (id, value) pairs for all originally-missing positions
df_s = pd.read_csv(FILLED_PATH)
df_s['datetime'] = pd.to_datetime(df_s['datetime'], format='%d-%m-%Y %H:%M')
df_o = pd.read_csv(DATASET_PATH)

records = []
for col in option_cols:
    for idx in df_o.index[df_o[col].isna()]:
        dt = df_s.loc[idx, 'datetime'].strftime('%d-%m-%Y %H:%M')
        records.append({'id': f'{dt}||{col}', 'value': df_s.loc[idx, col]})

sub = pd.DataFrame(records)
sub.to_csv(SUBMISSION_PATH, index=False)
print(f'Saved {SUBMISSION_PATH}')
print(f'Rows     : {len(sub)}')
print(f'Range    : [{sub.value.min():.5f}, {sub.value.max():.5f}]')
print(f'Negatives: {(sub.value < 0).sum()}')


## 7. Sanity Checks

In [ ]:
sub = pd.read_csv(SUBMISSION_PATH)
print(f'Rows         : {len(sub)}')
print(f'Value range  : [{sub["value"].min():.5f}, {sub["value"].max():.5f}]')
print(f'Negatives    : {(sub["value"] < 0).sum()}')
print(f'Floor hits   : {(sub["value"] <= 1e-4).sum()}')
print()
print(sub.head(8).to_string(index=False))


In [ ]:
# Visualise filled surface on expiry day (Jan 27) — shows totvar stabilising the smile
df_plot = pd.read_csv(FILLED_PATH)
df_plot['datetime'] = pd.to_datetime(df_plot['datetime'], format='%d-%m-%Y %H:%M')
exp_rows = df_plot[df_plot['datetime'].dt.date == pd.Timestamp('2026-01-27').date()]

fig, ax = plt.subplots(figsize=(14, 4))
for _, row in exp_rows.iterrows():
    ax.plot(ce_strikes, [row[c] for c in ce_cols], alpha=0.3, color='steelblue', lw=0.7)
ax.set_title('Filled CE IV smiles — Jan 27 (expiry day)  [all timestamps]')
ax.set_xlabel('Strike'); ax.set_ylabel('Implied Volatility')
plt.tight_layout(); plt.show()
